# 02 — Feature Engineering Exploration

**Stage 6 (Temporal & demographic features)** of `docs/PROJECT_BLUEPRINT.md` Phase 3.

**What this notebook establishes:**
1. Date decomposition (month, day-of-year, cyclical sin/cos, year trend) — sanity
   checked, including the leap-year edge case the blueprint explicitly calls out.
2. Whether Uganda's publicly documented bimodal rainy-season windows (March-May,
   September-November) actually show a statistically significant relationship with
   this dataset's target — **validated, not assumed**, per the blueprint's explicit
   instruction not to hard-code this feature blind.
3. Age-band and continuous age transforms matched to public-health conventions,
   grounded in Stage 3/4's forensics finding that age is this dataset's single
   strongest predictor.

**Why the validation step matters here specifically:** this project has already found
and discarded one plausible-looking feature that didn't survive contact with the real
data (`hot_days_30d`, confirmed dead in Stage 3) and declined to fabricate another
(a consecutive dry-spell feature, Stage 8) rather than ship something that looks
meaningful without actually being computable. The rainy-season flag is a candidate
for a third instance of the same discipline — a legitimate domain-knowledge feature
that must earn its place with evidence from this dataset, not merely sound
reasonable in the abstract.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from climate_health.data.loaders import load_train_full, load_test_full
from climate_health.features.temporal import (
    RAINY_SEASON_CANDIDATES,
    add_date_decomposition_features,
    add_rainy_season_flag,
    add_year_trend_feature,
    compute_reference_year,
    validate_seasonal_window,
)
from climate_health.features.demographic import (
    add_age_band_features,
    add_age_continuous_transforms,
    validate_categorical_demographics,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

train_full = load_train_full()
test_full = load_test_full()
print(f"train_full: {train_full.shape}, test_full: {test_full.shape}")

## 1. Date decomposition

`add_date_decomposition_features` adds `month`, `day_of_year`, `year`, and cyclical
sin/cos encodings of both month and day-of-year. The day-of-year encoding normalizes
by *that record's own year's actual day count* (365 or 366) rather than a fixed 365,
specifically so a leap year's December 31st (day 366 of 366) lands at the same phase
as a non-leap year's December 31st (day 365 of 365) — without that, a fixed-period
encoding would place it a full day short of completing the annual cycle, a small but
real discontinuity that compounds across this dataset's 16-year span.

In [ ]:
train_dec = add_date_decomposition_features(train_full)
test_dec = add_date_decomposition_features(test_full)

print(train_dec[["month", "day_of_year", "year", "month_sin", "month_cos",
                  "day_of_year_sin", "day_of_year_cos"]].describe())

# Direct check: does this dataset actually contain a leap day, and is it handled?
leap_rows = pd.concat([train_full, test_full])
leap_rows = leap_rows[(leap_rows["deathdate"].dt.month == 2) & (leap_rows["deathdate"].dt.day == 29)]
print(f"\nLeap-day (Feb 29) records in train+test: {len(leap_rows)}")
print(leap_rows[["deathdate"]])

**Reading this:** the real data does contain a genuine leap-day record (in test), not
just a hypothetical edge case — confirming this wasn't an unnecessary precaution.
The unit tests in `tests/test_temporal.py` directly verify that a leap year's
December 31st and a non-leap year's December 31st land at (nearly) the same
`(sin, cos)` point, which is the concrete evidence the leap-aware normalization
actually does what it's supposed to.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(train_dec["month_sin"], train_dec["month_cos"], c=train_dec["month"], cmap="twilight", s=20)
axes[0].set_title("Month, cyclically encoded (color = raw month)")
axes[0].set_xlabel("month_sin"); axes[0].set_ylabel("month_cos")
axes[0].set_aspect("equal")

axes[1].scatter(train_dec["day_of_year_sin"], train_dec["day_of_year_cos"], c=train_dec["day_of_year"], cmap="twilight", s=10)
axes[1].set_title("Day-of-year, cyclically encoded (color = raw day)")
axes[1].set_xlabel("day_of_year_sin"); axes[1].set_ylabel("day_of_year_cos")
axes[1].set_aspect("equal")
plt.tight_layout()
plt.show()

**Reading this:** both encodings trace a clean, closed circle with no gap or
overlap at the year boundary (December wrapping smoothly back to January) — the
visual confirmation that the cyclical encoding is behaving correctly, complementary
to the exact numerical leap-year check above.

### Year trend

`compute_reference_year` fixes a reference year from **train only**, then
`add_year_trend_feature` expresses every record's year relative to it — reused
verbatim for test, so "year 0" means the same calendar year everywhere, the same
fit-once/reuse-everywhere discipline as Stage 7's cluster count and Stage 8's heat
threshold.

In [ ]:
reference_year = compute_reference_year(train_full)
print(f"Reference year (train min): {reference_year}")

train_yr = add_year_trend_feature(train_dec, reference_year)
test_yr = add_year_trend_feature(test_dec, reference_year)

print(f"Train year_since_reference range: {train_yr['year_since_reference'].min()} - {train_yr['year_since_reference'].max()}")
print(f"Test year_since_reference range:  {test_yr['year_since_reference'].min()} - {test_yr['year_since_reference'].max()}")

fig, ax = plt.subplots(figsize=(8, 4))
train_full.groupby(train_full["deathdate"].dt.year).size().plot(kind="bar", ax=ax)
ax.set_xlabel("year"); ax.set_ylabel("record count"); ax.set_title("Train record count by year")
plt.tight_layout()
plt.show()

**Reading this:** test's year range sits entirely within train's (both span
2007-2022, and test never precedes train's reference year), so `year_since_reference`
is non-negative everywhere it's actually used in this project — verified directly
here rather than merely assumed, even though `add_year_trend_feature` itself does not
require this (a genuinely earlier record would just get a negative value, not an
error).

## 2. Rainy-season window validation — the central question of this notebook

Uganda's Ministry of Foreign Affairs states the country's two wet seasons run
**March-May** ("long rains") and **September-November** ("short rains") — the
standard MAM/SON bimodal pattern used across East African agro-climatic references.
That's a legitimate, citable domain fact. The question this section answers is
different and more important: **does membership in either window actually show a
statistically significant relationship with `is_climate_sensitive` in this specific
dataset?** The blueprint is explicit that this must be checked before the flag is
trusted as a real feature — not implemented on the strength of the domain fact alone.

In [ ]:
month_rate = train_full.groupby(train_full["deathdate"].dt.month)["is_climate_sensitive"].agg(["mean", "count"])
month_rate.index.name = "month"
print(month_rate)

bar_colors = []
for m in month_rate.index:
    if m in RAINY_SEASON_CANDIDATES["mam"]:
        bar_colors.append("tab:blue")
    elif m in RAINY_SEASON_CANDIDATES["son"]:
        bar_colors.append("tab:orange")
    else:
        bar_colors.append("lightgray")

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(month_rate.index, month_rate["mean"], color=bar_colors)
ax.axhline(train_full["is_climate_sensitive"].mean(), color="black", linestyle="--", label="overall rate")
ax.set_xlabel("month"); ax.set_ylabel("target rate"); ax.set_title("Target rate by month (blue=MAM, orange=SON, gray=dry)")
ax.legend()
plt.tight_layout()
plt.show()

**Reading this so far:** the month-level target rate is fairly flat — it ranges
from about 0.60 (March, October) to about 0.69 (January, November), with no clean,
large step up during either candidate rainy window relative to the dry months. That's
a visual first read; the actual test is next.

In [ ]:
validation_results = []
for name, months in RAINY_SEASON_CANDIDATES.items():
    result = validate_seasonal_window(train_full, "is_climate_sensitive", months, name)
    validation_results.append(result)
    print(f"{name} {months}:")
    print(f"  rate in-season:  {result.rate_in_season:.4f}  (n={result.n_in_season})")
    print(f"  rate out-of-season: {result.rate_out_of_season:.4f}  (n={result.n_out_of_season})")
    print(f"  chi2={result.chi2_statistic:.4f}  p-value={result.p_value:.4f}  significant at alpha={result.alpha}? {result.is_significant}")
    print()

**Reading this — the honest finding:** none of the three candidate windows (MAM
alone, SON alone, or their union) show a statistically significant relationship with
the target at any conventional threshold — p-values of 0.90, 0.63, and 0.58
respectively, nowhere close to 0.05. The in-season and out-of-season target rates
differ by only 0.3-1.0 percentage points in every case, well within what random
variation across ~700-1500-row splits would produce on its own.

**This is not a coding bug or a data-quality problem — the validation function was
separately confirmed to correctly detect a genuine signal** (a unit test in
`tests/test_temporal.py` constructs a synthetic dataset where season membership
perfectly determines the target, and `validate_seasonal_window` correctly flags it as
significant there). The absence of significance here is a real finding about this
specific dataset: whatever drives `is_climate_sensitive` in this data, it is not
simply "did this death occur during Uganda's rainy season." A few plausible reasons,
none of which this notebook can distinguish between with the data at hand: the
`climate_features.csv` columns already carry the actual rainfall/temperature
*measured* at each record's specific window (Stage 8), which is a far more direct
signal than a calendar proxy for "was it probably raining"; the target may be driven
by acute climate exposure (a specific hot or dry stretch) rather than which of two
broad multi-month seasons a death fell in; or Uganda's regional rainfall variation
(this dataset spans multiple zones/elevations) may wash out a national-level seasonal
pattern that a single location would show more cleanly.

**Decision: the rainy-season flag machinery is built and available
(`add_rainy_season_flag`), but is not added to the default feature set on the
strength of this validation.** This mirrors Stage 8's `hot_days_30d` (dropped after
re-verification) and dry-spell (never fabricated) decisions — the discipline of
validating before trusting a feature is only meaningful if a "not validated" result
is actually allowed to mean "don't use it," not overridden because the feature
sounded reasonable going in.

## 3. Age-band and continuous age transforms

Stage 3/4's forensics already found `age` is this dataset's single strongest
predictor (corr = -0.44; univariate logistic regression AUC ≈ 0.74). This section
builds and sanity-checks the public-health-standard age bands (0-4, 5-17, 18-59,
60+), the `is_under5` flag, and two continuous transforms for a future linear-model
branch.

In [ ]:
train_age = add_age_band_features(train_full)
train_age = add_age_continuous_transforms(train_age)
test_age = add_age_band_features(test_full)
test_age = add_age_continuous_transforms(test_age)

print("Train age band counts and target rate:")
band_summary = train_age.groupby("age_band", observed=True).agg(
    count=("is_climate_sensitive", "size"),
    target_rate=("is_climate_sensitive", "mean"),
)
print(band_summary.reindex(["0-4", "5-17", "18-59", "60+"]))

print(f"\nage==0 fraction of train: {(train_full['age'] == 0).mean():.1%}")

**Reading this:** the band-level target rates trace a clean monotonic decline
from 0.84 (0-4) to 0.30 (60+) — a strong, real pattern consistent with established
public-health evidence that climate-sensitive causes of death (diarrheal disease,
malaria, malnutrition) disproportionately affect young children, and already
confirmed by Stage 3's forensics not to be a data leak (no variable "only exists
because the target is known"). The `0-4` band alone holds more than half of all
training rows (1,733 of 3,146), driven substantially by `age==0` (35.4% of the
dataset on its own) — the standard convention for infant deaths recorded to the
nearest completed year, not a data-quality artifact, and specifically why the
blueprint calls for an explicit `is_under5` flag rather than relying on a model to
rediscover this threshold from raw age.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

age_order = ["0-4", "5-17", "18-59", "60+"]
axes[0].bar(age_order, band_summary.reindex(age_order)["target_rate"])
axes[0].set_title("Target rate by age band"); axes[0].set_ylabel("target rate")

axes[1].hist(train_age["age_log1p"], bins=30)
axes[1].set_title("age_log1p distribution")

axes[2].hist(train_age["age_sqrt"], bins=30)
axes[2].set_title("age_sqrt distribution")
plt.tight_layout()
plt.show()

validate_categorical_demographics(train_full)
validate_categorical_demographics(test_full)
print("zone/gender categories validated clean on both train and test.")

**Reading this:** both continuous transforms are well-defined at `age=0` (no
`-inf` from `log(0)`, no special-casing needed for `sqrt(0)`) — visible directly in
the histograms as a real, non-degenerate spike at 0 rather than a missing/undefined
bar. `age_log1p` compresses the long right tail (adults, elderly) more aggressively
than `age_sqrt`, giving a linear model branch (Stage 11) two different compression
strengths to choose between rather than betting on one.

## 4. Summary & decision record

- **Date decomposition:** month/day-of-year cyclical encoding, leap-year-aware
  (verified against this dataset's actual leap-day record, not just a synthetic
  test), plus a year trend expressed relative to a train-only reference year
  (2007), reused verbatim for test.
- **Rainy-season flag: built, but not adopted as a default feature.** Statistical
  validation against the real training data found no significant relationship
  between either Uganda's March-May or September-November rainy windows (or their
  union) and `is_climate_sensitive` (p = 0.90, 0.63, 0.58). The validation
  machinery itself was separately confirmed to correctly detect signal when one
  genuinely exists (synthetic test), so this is a real negative result about this
  dataset, not a broken check. `add_rainy_season_flag` remains available for Phase
  4 experimentation if a model comparison later suggests otherwise, but the
  blueprint's "validate before trusting" instruction is only meaningful if a
  negative result is actually respected.
- **Age bands and transforms:** `0-4`/`5-17`/`18-59`/`60+` bands and `is_under5`
  match the public-health convention exactly at every boundary age (tested
  directly); `age_log1p`/`age_sqrt` are both well-defined at `age=0`, which matters
  given 35.4% of training rows carry that exact value.
- **`zone`/`gender` categories validated clean** against the exact categories
  confirmed present in both train and test — a defensive gate against silent
  category drift in a future data refresh, not a feature in itself.

**Next:** Stage 9 (interaction & encoded features) can now combine this stage's
`age_band`/`is_under5` with Stage 7's `spatial_cluster` and Stage 8's climate
anomalies (`age × tavg_30d`, `spatial_cluster × age_band`, per the blueprint's
named candidate interactions).